In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("../data/synthetic")

print("=" * 70)
print("02.30 - REFERENTIAL INTEGRITY VALIDATION")
print("=" * 70)

# Load master reference keys
customers = pd.read_csv(
    BASE_DIR / "customers_master.csv",
    usecols=["customer_id"],
    dtype=str
)

products = pd.read_csv(
    BASE_DIR / "products_master.csv",
    usecols=["product_id"],
    dtype=str
)

stores = pd.read_csv(
    BASE_DIR / "stores_master.csv",
    usecols=["store_id"],
    dtype=str
)

salespersons = pd.read_csv(
    BASE_DIR / "salespersons_master.csv",
    usecols=["salesperson_id"],
    dtype=str
)

promotions = pd.read_csv(
    BASE_DIR / "promotions_master.csv",
    usecols=["promotion_id"],
    dtype=str
)

# Create reference key sets
customer_keys = set(customers["customer_id"].dropna())
product_keys = set(products["product_id"].dropna())
store_keys = set(stores["store_id"].dropna())
salesperson_keys = set(salespersons["salesperson_id"].dropna())
promotion_keys = set(promotions["promotion_id"].dropna())

print("\nREFERENCE KEY COUNTS")
print("-" * 70)
print(f"Customer IDs       : {len(customer_keys):,}")
print(f"Product IDs        : {len(product_keys):,}")
print(f"Store IDs          : {len(store_keys):,}")
print(f"Salesperson IDs    : {len(salesperson_keys):,}")
print(f"Promotion IDs      : {len(promotion_keys):,}")

print("\n02.30 Step 1 preparation: PASSED")

02.30 - REFERENTIAL INTEGRITY VALIDATION

REFERENCE KEY COUNTS
----------------------------------------------------------------------
Customer IDs       : 10,000
Product IDs        : 2,500
Store IDs          : 60
Salesperson IDs    : 500
Promotion IDs      : 72

02.30 Step 1 preparation: PASSED


In [3]:
print("=" * 70)
print("02.30 - STEP 2 - SALES REFERENTIAL INTEGRITY")
print("=" * 70)

sales_files = sorted(BASE_DIR.glob("sales_*.csv"))

total_rows = 0

invalid_customers = 0
invalid_products = 0
invalid_stores = 0
invalid_salespersons = 0
invalid_promotions = 0

blank_customers = 0
blank_salespersons = 0

for file in sales_files:

    df = pd.read_csv(
        file,
        usecols=[
            "customer_id",
            "product_id",
            "store_id",
            "salesperson_id",
            "promotion_id"
        ],
        dtype="string"
    )

    total_rows += len(df)

    # -----------------------------
    # Customer validation
    # -----------------------------
    customer_blank = (
        df["customer_id"].isna()
        | df["customer_id"].str.strip().eq("")
    )

    blank_customers += customer_blank.sum()

    invalid_customers += (
        ~customer_blank
        & ~df["customer_id"].isin(customer_keys)
    ).sum()

    # -----------------------------
    # Product validation
    # -----------------------------
    invalid_products += (
        df["product_id"].notna()
        & ~df["product_id"].isin(product_keys)
    ).sum()

    # -----------------------------
    # Store validation
    # -----------------------------
    invalid_stores += (
        df["store_id"].notna()
        & ~df["store_id"].isin(store_keys)
    ).sum()

    # -----------------------------
    # Salesperson validation
    # -----------------------------
    salesperson_blank = (
        df["salesperson_id"].isna()
        | df["salesperson_id"].str.strip().eq("")
    )

    blank_salespersons += salesperson_blank.sum()

    invalid_salespersons += (
        ~salesperson_blank
        & ~df["salesperson_id"].isin(salesperson_keys)
    ).sum()

    # -----------------------------
    # Promotion validation
    # -----------------------------
    promotion_present = (
        df["promotion_id"].notna()
        & df["promotion_id"].str.strip().ne("")
    )

    invalid_promotions += (
        promotion_present
        & ~df["promotion_id"].isin(promotion_keys)
    ).sum()


print(f"\nSales files checked       : {len(sales_files)}")
print(f"Total sales rows checked : {total_rows:,}")

print("\nBLANK / NULL VALUES")
print("-" * 70)
print(f"Blank customer IDs       : {blank_customers:,}")
print(f"Blank salesperson IDs    : {blank_salespersons:,}")

print("\nINVALID FOREIGN KEY COUNTS")
print("-" * 70)
print(f"Invalid customer IDs     : {invalid_customers:,}")
print(f"Invalid product IDs      : {invalid_products:,}")
print(f"Invalid store IDs        : {invalid_stores:,}")
print(f"Invalid salesperson IDs  : {invalid_salespersons:,}")
print(f"Invalid promotion IDs    : {invalid_promotions:,}")

if (
    invalid_customers == 0
    and invalid_products == 0
    and invalid_stores == 0
    and invalid_salespersons == 0
    and invalid_promotions == 0
):
    print("\n02.30 Step 2 validation: PASSED")
else:
    print("\n02.30 Step 2 validation: NEEDS REVIEW")

02.30 - STEP 2 - SALES REFERENTIAL INTEGRITY

Sales files checked       : 32
Total sales rows checked : 2,118,200

BLANK / NULL VALUES
----------------------------------------------------------------------
Blank customer IDs       : 63,501
Blank salesperson IDs    : 529,575

INVALID FOREIGN KEY COUNTS
----------------------------------------------------------------------
Invalid customer IDs     : 0
Invalid product IDs      : 0
Invalid store IDs        : 0
Invalid salesperson IDs  : 0
Invalid promotion IDs    : 0

02.30 Step 2 validation: PASSED


In [4]:
print("=" * 70)
print("02.30 - STEP 3 - RETURNS REFERENTIAL INTEGRITY")
print("=" * 70)

# ---------------------------------------
# Build valid sales_line_id reference set
# ---------------------------------------

sales_line_keys = set()

for file in sorted(BASE_DIR.glob("sales_*.csv")):

    df = pd.read_csv(
        file,
        usecols=["sales_line_id"],
        dtype="string"
    )

    sales_line_keys.update(
        df["sales_line_id"]
        .dropna()
        .str.strip()
    )

print(f"\nValid sales_line_ids available : {len(sales_line_keys):,}")


# ---------------------------------------
# Validate returns
# ---------------------------------------

returns_files = sorted(BASE_DIR.glob("returns_*.csv"))

total_return_rows = 0
blank_sales_line_ids = 0
invalid_sales_line_ids = 0

for file in returns_files:

    df = pd.read_csv(
        file,
        usecols=["sales_line_id"],
        dtype="string"
    )

    total_return_rows += len(df)

    blank_mask = (
        df["sales_line_id"].isna()
        | df["sales_line_id"].str.strip().eq("")
    )

    blank_sales_line_ids += blank_mask.sum()

    invalid_sales_line_ids += (
        ~blank_mask
        & ~df["sales_line_id"].isin(sales_line_keys)
    ).sum()


# ---------------------------------------
# Final result
# ---------------------------------------

print("\nRETURNS VALIDATION")
print("-" * 70)

print(f"Returns files checked       : {len(returns_files)}")
print(f"Total return rows checked  : {total_return_rows:,}")

print("\nREFERENCE KEY RESULTS")
print("-" * 70)

print(f"Blank sales_line_ids       : {blank_sales_line_ids:,}")
print(f"Invalid sales_line_ids     : {invalid_sales_line_ids:,}")

if invalid_sales_line_ids == 0:

    print("\n02.30 Step 3 validation: PASSED")

else:

    print("\n02.30 Step 3 validation: NEEDS REVIEW")

02.30 - STEP 3 - RETURNS REFERENTIAL INTEGRITY

Valid sales_line_ids available : 2,118,200

RETURNS VALIDATION
----------------------------------------------------------------------
Returns files checked       : 32
Total return rows checked  : 118,400

REFERENCE KEY RESULTS
----------------------------------------------------------------------
Blank sales_line_ids       : 0
Invalid sales_line_ids     : 0

02.30 Step 3 validation: PASSED


In [6]:
print("=" * 70)
print("02.30 - STEP 4 - TARGETS REFERENTIAL INTEGRITY")
print("=" * 70)

# ---------------------------------------
# Load required reference files directly
# ---------------------------------------

stores_master = pd.read_csv(
    BASE_DIR / "stores_master.csv",
    dtype="string"
)

products_master = pd.read_csv(
    BASE_DIR / "products_master.csv",
    dtype="string"
)

targets = pd.read_csv(
    BASE_DIR / "targets.csv",
    dtype="string"
)

print(f"\nTarget rows checked : {len(targets):,}")


# ---------------------------------------
# Create reference key sets
# ---------------------------------------

target_store_keys = set(
    stores_master["store_id"]
    .dropna()
    .str.strip()
)

valid_categories = set(
    products_master["category"]
    .dropna()
    .str.strip()
)


# ---------------------------------------
# Target store validation
# ---------------------------------------

blank_store_ids = (
    targets["store_id"].isna()
    | targets["store_id"].str.strip().eq("")
).sum()

invalid_target_stores = (
    targets["store_id"].notna()
    & targets["store_id"].str.strip().ne("")
    & ~targets["store_id"].str.strip().isin(target_store_keys)
).sum()


# ---------------------------------------
# Target category validation
# ---------------------------------------

blank_categories = (
    targets["category"].isna()
    | targets["category"].str.strip().eq("")
).sum()

invalid_target_categories = (
    targets["category"].notna()
    & targets["category"].str.strip().ne("")
    & ~targets["category"].str.strip().isin(valid_categories)
).sum()


# ---------------------------------------
# Results
# ---------------------------------------

print("\nTARGETS REFERENCE VALIDATION")
print("-" * 70)

print(f"Blank store IDs            : {blank_store_ids:,}")
print(f"Invalid store IDs          : {invalid_target_stores:,}")
print(f"Blank categories           : {blank_categories:,}")
print(f"Invalid categories         : {invalid_target_categories:,}")


if (
    blank_store_ids == 0
    and invalid_target_stores == 0
    and blank_categories == 0
    and invalid_target_categories == 0
):
    print("\n02.30 Step 4 validation: PASSED")
else:
    print("\n02.30 Step 4 validation: NEEDS REVIEW")

02.30 - STEP 4 - TARGETS REFERENTIAL INTEGRITY

Target rows checked : 8,980

TARGETS REFERENCE VALIDATION
----------------------------------------------------------------------
Blank store IDs            : 0
Invalid store IDs          : 0
Blank categories           : 0
Invalid categories         : 0

02.30 Step 4 validation: PASSED


In [7]:
print("=" * 70)
print("02.30 - STEP 5 - FINAL REFERENTIAL INTEGRITY SUMMARY")
print("=" * 70)

print("\nSALES")
print("-" * 70)
print(f"Sales rows checked          : {total_rows:,}")
print(f"Invalid customer IDs        : {invalid_customers:,}")
print(f"Invalid product IDs         : {invalid_products:,}")
print(f"Invalid store IDs           : {invalid_stores:,}")
print(f"Invalid salesperson IDs     : {invalid_salespersons:,}")
print(f"Invalid promotion IDs       : {invalid_promotions:,}")

print("\nRETURNS")
print("-" * 70)
print(f"Returns rows checked        : {total_return_rows:,}")
print(f"Blank sales_line_ids        : {blank_sales_line_ids:,}")
print(f"Invalid sales_line_ids      : {invalid_sales_line_ids:,}")

print("\nTARGETS")
print("-" * 70)
print(f"Target rows checked         : {len(targets):,}")
print(f"Blank store IDs             : {blank_store_ids:,}")
print(f"Invalid store IDs           : {invalid_target_stores:,}")
print(f"Blank categories            : {blank_categories:,}")
print(f"Invalid categories          : {invalid_target_categories:,}")


all_referential_integrity_valid = (
    invalid_customers == 0
    and invalid_products == 0
    and invalid_stores == 0
    and invalid_salespersons == 0
    and invalid_promotions == 0
    and invalid_sales_line_ids == 0
    and blank_store_ids == 0
    and invalid_target_stores == 0
    and blank_categories == 0
    and invalid_target_categories == 0
)

print("\n" + "=" * 70)

if all_referential_integrity_valid:
    print("02.30 Step 5 validation: PASSED")
else:
    print("02.30 Step 5 validation: NEEDS REVIEW")

02.30 - STEP 5 - FINAL REFERENTIAL INTEGRITY SUMMARY

SALES
----------------------------------------------------------------------
Sales rows checked          : 2,118,200
Invalid customer IDs        : 0
Invalid product IDs         : 0
Invalid store IDs           : 0
Invalid salesperson IDs     : 0
Invalid promotion IDs       : 0

RETURNS
----------------------------------------------------------------------
Returns rows checked        : 118,400
Blank sales_line_ids        : 0
Invalid sales_line_ids      : 0

TARGETS
----------------------------------------------------------------------
Target rows checked         : 8,980
Blank store IDs             : 0
Invalid store IDs           : 0
Blank categories            : 0
Invalid categories          : 0

02.30 Step 5 validation: PASSED


In [8]:
import pandas as pd
from pathlib import Path

REPORT_FILE = Path("../data/synthetic/referential_integrity_validation_report.csv")

report = pd.DataFrame([
    {
        "dataset": "sales",
        "rows_checked": total_rows,
        "invalid_customer_ids": invalid_customers,
        "invalid_product_ids": invalid_products,
        "invalid_store_ids": invalid_stores,
        "invalid_salesperson_ids": invalid_salespersons,
        "invalid_promotion_ids": invalid_promotions,
        "status": "VALID"
    },
    {
        "dataset": "returns",
        "rows_checked": total_return_rows,
        "blank_sales_line_ids": blank_sales_line_ids,
        "invalid_sales_line_ids": invalid_sales_line_ids,
        "status": "VALID"
    },
    {
        "dataset": "targets",
        "rows_checked": len(targets),
        "blank_store_ids": blank_store_ids,
        "invalid_store_ids": invalid_target_stores,
        "blank_categories": blank_categories,
        "invalid_categories": invalid_target_categories,
        "status": "VALID"
    }
])

report.to_csv(REPORT_FILE, index=False)

print("=" * 70)
print("02.30 - STEP 6 - SAVE REFERENTIAL INTEGRITY REPORT")
print("=" * 70)

print(f"\nReport saved:")
print(REPORT_FILE)

print(f"\nRows in report: {len(report)}")
print(f"All statuses VALID: {(report['status'] == 'VALID').all()}")

if (report["status"] == "VALID").all():
    print("\n02.30 Step 6 validation: PASSED")
else:
    print("\n02.30 Step 6 validation: NEEDS REVIEW")

02.30 - STEP 6 - SAVE REFERENTIAL INTEGRITY REPORT

Report saved:
..\data\synthetic\referential_integrity_validation_report.csv

Rows in report: 3
All statuses VALID: True

02.30 Step 6 validation: PASSED
